# Person ReID Pipeline — Market-1501

End-to-end notebook combining dataset loading, model definition, training, evaluation, and plotting.

**Loss combination (Bag of Tricks):** `L_total = L_id + L_triplet + λ * L_center`  
**LR schedule:** 10-epoch linear warmup → cosine annealing  
**Sampling:** P×K (P identities × K images per batch)  
**Evaluation:** Rank-1 + mAP, Market-1501 protocol, optional k-reciprocal re-ranking

## 1. Imports

In [ ]:
import os
import re
import time
import random
import logging
from collections import defaultdict
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T
from torchvision import models

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

## 2. Configuration

Edit these variables instead of passing command-line flags.

In [ ]:
# --- Paths ---
DATA_ROOT  = "./market_1501_data"   # dir containing processed/, keypoints/, skeleton_images/
OUTPUT_DIR = "./checkpoints"
LOG_PATH   = "./train.log"          # written during training; used by the plot cell

# --- Training ---
EPOCHS        = 120
WARMUP_EPOCHS = 10
NUM_PIDS      = 8       # P: identities per batch
NUM_IMGS      = 8       # K: images per identity
EVAL_BS       = 64
LR            = 3.5e-4
CENTER_LR     = 0.5
CENTER_WEIGHT = 5e-4
EVAL_EVERY    = 5
SEED          = 42

# --- Image size ---
IMG_H, IMG_W = 256, 128

# --- Modality flags ---
USE_KEYPOINTS = False   # feed (33,3) MediaPipe keypoints alongside RGB
USE_IBN       = False   # ResNet50-IBN-a backbone (better cross-camera)
USE_SKELETON  = False   # legacy: rendered skeleton images instead of RGB
INCLUDE_NO_DETECTION = False

# --- Re-ranking at eval time ---
USE_RERANK    = False
RERANK_K1     = 20
RERANK_K2     = 6
RERANK_LAMBDA = 0.3

# --- Derived ---
USE_RGB  = not USE_SKELETON
IMG_SIZE = (IMG_H, IMG_W)

torch.manual_seed(SEED)
random.seed(SEED)

device = (
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)
print(f"Device: {device}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Dataset

RGB crops (default) or skeleton images, paired with identity labels and optional MediaPipe keypoints.

**Directory layout** (produced by `preprocess.py` and `pose_estimation.py`):
```
processed/{split}/{pid:04d}/{filename}.jpg        ← RGB input (default)
skeleton_images/{split}/{pid:04d}/{stem}.jpg      ← legacy skeleton input
keypoints/{split}/{pid:04d}/{stem}.npy            ← (33,3) keypoints
```

In [ ]:
# ---------------------------------------------------------------------------
# Keypoint normalization
# ---------------------------------------------------------------------------

LEFT_HIP, RIGHT_HIP           = 23, 24
LEFT_SHOULDER, RIGHT_SHOULDER = 11, 12


def normalize_keypoints(kp: np.ndarray) -> np.ndarray:
    """
    Make keypoints translation- and scale-invariant.
    Centers on hip midpoint and divides by torso length.
    """
    if not np.any(kp):
        return kp
    out = kp.copy()
    hip_mid = (kp[LEFT_HIP, :2] + kp[RIGHT_HIP, :2]) / 2
    sh_mid  = (kp[LEFT_SHOULDER, :2] + kp[RIGHT_SHOULDER, :2]) / 2
    torso   = np.linalg.norm(sh_mid - hip_mid) + 1e-6
    out[:, :2] = (kp[:, :2] - hip_mid) / torso
    return out


# ---------------------------------------------------------------------------
# Transforms
# ---------------------------------------------------------------------------

def build_transforms(split: str, img_size=(256, 128), use_rgb: bool = True):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    if split == "train":
        ops = [T.Resize(img_size)]
        if use_rgb:
            ops.append(T.ColorJitter(0.2, 0.2, 0.2, 0.1))
        ops += [
            T.RandomHorizontalFlip(p=0.5),
            T.Pad(10),
            T.RandomCrop(img_size),
            T.ToTensor(),
            T.Normalize(mean, std),
            T.RandomErasing(
                p     = 0.5  if use_rgb else 0.1,
                scale = (0.02, 0.25) if use_rgb else (0.02, 0.08),
            ),
        ]
        return T.Compose(ops)
    else:
        return T.Compose([
            T.Resize(img_size),
            T.ToTensor(),
            T.Normalize(mean, std),
        ])


# ---------------------------------------------------------------------------
# Dataset class
# ---------------------------------------------------------------------------

class SkeletonReIDDataset(Dataset):
    """
    Loads RGB crops (default) or skeleton images, paired with identity
    labels and optional MediaPipe keypoint vectors.
    """

    def __init__(self,
                 data_root: str,
                 split: str,
                 transform=None,
                 use_rgb: bool = True,
                 use_keypoints: bool = False,
                 skip_no_detection: bool = False):

        self.data_root         = data_root
        self.split             = split
        self.transform         = transform
        self.use_rgb           = use_rgb
        self.use_keypoints     = use_keypoints
        self.skip_no_detection = skip_no_detection

        self.processed_dir = os.path.join(data_root, "processed",       split)
        self.skeleton_dir  = os.path.join(data_root, "skeleton_images", split)
        self.keypoint_dir  = os.path.join(data_root, "keypoints",       split)

        self.samples   = []
        self.pid2label = {}
        self._load_samples()

    def _load_samples(self):
        if not os.path.isdir(self.processed_dir):
            raise FileNotFoundError(
                f"Processed split not found: {self.processed_dir}\n"
                "Run preprocess.py first."
            )
        if not self.use_rgb and not os.path.isdir(self.skeleton_dir):
            raise FileNotFoundError(
                f"Skeleton directory not found: {self.skeleton_dir}\n"
                "Run pose_estimation.py first, or pass use_rgb=True."
            )

        skipped_no_det  = 0
        skipped_missing = 0

        pids = sorted(os.listdir(self.processed_dir))
        for label_idx, pid_str in enumerate(pids):
            pid_proc_dir = os.path.join(self.processed_dir, pid_str)
            if not os.path.isdir(pid_proc_dir):
                continue
            self.pid2label[pid_str] = label_idx

            for fname in sorted(os.listdir(pid_proc_dir)):
                if not fname.lower().endswith((".jpg", ".png", ".jpeg")):
                    continue

                stem = os.path.splitext(fname)[0]

                if self.use_rgb:
                    img_path = os.path.join(self.processed_dir, pid_str, fname)
                else:
                    img_path = os.path.join(self.skeleton_dir, pid_str, fname)
                    if not os.path.isfile(img_path):
                        img_path = os.path.join(self.skeleton_dir, pid_str, stem + ".png")

                if not os.path.isfile(img_path):
                    skipped_missing += 1
                    continue

                kp_path = os.path.join(self.keypoint_dir, pid_str, stem + ".npy")

                if self.skip_no_detection and os.path.isfile(kp_path):
                    kp = np.load(kp_path)
                    if not np.any(kp):
                        skipped_no_det += 1
                        continue

                try:
                    cam_id = int(fname.split("_c")[1][0]) - 1
                except (IndexError, ValueError):
                    cam_id = 0

                self.samples.append((img_path, kp_path, label_idx, cam_id))

        modality = "RGB" if self.use_rgb else "skeleton"
        print(
            f"[{self.split}/{modality}] {len(self.samples)} samples | "
            f"{len(self.pid2label)} identities | "
            f"skipped (no pose): {skipped_no_det} | "
            f"skipped (missing file): {skipped_missing}"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, kp_path, pid_label, cam_id = self.samples[idx]

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        keypoints = torch.zeros(99, dtype=torch.float32)
        if self.use_keypoints and os.path.isfile(kp_path):
            kp = np.load(kp_path).astype(np.float32)
            kp = normalize_keypoints(kp)
            keypoints = torch.from_numpy(kp.flatten())

        return {
            "img":       img,
            "pid":       torch.tensor(pid_label, dtype=torch.long),
            "cam":       torch.tensor(cam_id,    dtype=torch.long),
            "keypoints": keypoints,
            "path":      img_path,
        }

## 4. Model

ResNet50 (optionally IBN-a) with a BN-neck for ReID. Supports optional MediaPipe keypoint fusion.

In [ ]:
# ---------------------------------------------------------------------------
# Weight initialisers
# ---------------------------------------------------------------------------

def _init_kaiming(m):
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)
    elif isinstance(m, nn.BatchNorm1d):
        nn.init.constant_(m.weight, 1.0)
        nn.init.constant_(m.bias,   0.0)


def _init_classifier(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, std=0.001)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)


# ---------------------------------------------------------------------------
# Backbone selection
# ---------------------------------------------------------------------------

def _build_backbone(use_ibn: bool, pretrained: bool):
    if use_ibn:
        try:
            backbone = torch.hub.load(
                "XingangPan/IBN-Net",
                "resnet50_ibn_a",
                pretrained=pretrained,
            )
            print("[Model] Using ResNet50-IBN-a backbone")
            return backbone
        except Exception as e:
            print(f"[Model] IBN-Net load failed ({e}); falling back to ResNet50")

    weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
    print("[Model] Using ResNet50 backbone")
    return models.resnet50(weights=weights)


# ---------------------------------------------------------------------------
# Main model
# ---------------------------------------------------------------------------

class GaitReIDNet(nn.Module):
    """
    ResNet50 (optionally IBN-a) + BN-neck for ReID.

    Args:
        num_classes   : number of training identities
        feat_dim      : backbone output channels (2048 for ResNet50)
        last_stride   : 1 keeps more spatial resolution in layer4 (BoT paper)
        pretrained    : load ImageNet weights for the backbone
        use_keypoints : if True, accepts (B, 99) keypoint vector and fuses it into the embedding
        use_ibn       : if True, use ResNet50-IBN-a instead of ResNet50
    """

    def __init__(self,
                 num_classes:   int,
                 feat_dim:      int  = 2048,
                 last_stride:   int  = 1,
                 pretrained:    bool = True,
                 use_keypoints: bool = False,
                 use_ibn:       bool = False):
        super().__init__()
        self.feat_dim      = feat_dim
        self.num_classes   = num_classes
        self.use_keypoints = use_keypoints

        backbone = _build_backbone(use_ibn, pretrained)

        if last_stride == 1:
            backbone.layer4[0].conv2.stride         = (1, 1)
            backbone.layer4[0].downsample[0].stride = (1, 1)

        self.backbone = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4,
        )

        self.gap = nn.AdaptiveAvgPool2d(1)

        if use_keypoints:
            self.kp_encoder = nn.Sequential(
                nn.Linear(99, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Linear(256, 256),
            )
            self.fusion = nn.Linear(feat_dim + 256, feat_dim)
            self.fusion.apply(_init_kaiming)

        self.bottleneck = nn.BatchNorm1d(feat_dim)
        self.bottleneck.bias.requires_grad_(False)
        self.bottleneck.apply(_init_kaiming)

        self.classifier = nn.Linear(feat_dim, num_classes, bias=False)
        self.classifier.apply(_init_classifier)

    def forward(self, x, keypoints=None):
        feat_map  = self.backbone(x)
        pool_feat = self.gap(feat_map).flatten(1)

        if self.use_keypoints and keypoints is not None:
            kp_feat   = self.kp_encoder(keypoints)
            pool_feat = self.fusion(torch.cat([pool_feat, kp_feat], dim=1))

        bn_feat = self.bottleneck(pool_feat)

        if not self.training:
            return F.normalize(bn_feat, dim=1)

        logits = self.classifier(bn_feat)
        return logits, pool_feat, bn_feat

    def get_embedding(self, x, keypoints=None):
        """Always returns L2-normalized BN feature regardless of training state."""
        was_training = self.training
        self.eval()
        with torch.no_grad():
            emb = self.forward(x, keypoints)
        if was_training:
            self.train()
        return emb

## 5. Losses & Sampler

In [ ]:
class LabelSmoothingCE(nn.Module):
    """Cross-entropy with label smoothing (epsilon=0.1)."""
    def __init__(self, num_classes: int, epsilon: float = 0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon     = epsilon
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, logits, targets):
        log_probs = self.log_softmax(logits)
        with torch.no_grad():
            smooth = torch.full_like(log_probs,
                                     self.epsilon / (self.num_classes - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.epsilon)
        return -(smooth * log_probs).sum(dim=1).mean()


class BatchHardTripletLoss(nn.Module):
    """Batch-hard triplet loss: mines hardest positive/negative per anchor."""
    def __init__(self, margin: float = 0.3, soft: bool = True):
        super().__init__()
        self.margin = margin
        self.soft   = soft

    def _pairwise_dist(self, x):
        dot = x @ x.t()
        sq  = dot.diag().unsqueeze(1)
        return (sq + sq.t() - 2 * dot).clamp(min=1e-12).sqrt()

    def forward(self, embeddings, labels):
        dist     = self._pairwise_dist(embeddings)
        same_pid = labels.unsqueeze(1) == labels.unsqueeze(0)
        diff_pid = ~same_pid

        ap = dist.clone(); ap[diff_pid] = -1e9
        an = dist.clone(); an[same_pid] =  1e9

        hardest_pos = ap.max(dim=1).values
        hardest_neg = an.min(dim=1).values

        if self.soft:
            return torch.log1p(torch.exp(hardest_pos - hardest_neg)).mean()
        return torch.relu(hardest_pos - hardest_neg + self.margin).mean()


class CenterLoss(nn.Module):
    """Center loss — tightens intra-class clusters in feature space."""
    def __init__(self, num_classes: int, feat_dim: int = 2048):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

    def forward(self, features, labels):
        centers_batch = self.centers[labels]
        return ((features - centers_batch) ** 2).sum(dim=1).mean() / 2


class PKSampler(Sampler):
    """Yields indices such that each batch has exactly P identities × K images."""
    def __init__(self, dataset, num_pids: int = 8, num_imgs: int = 8):
        self.P = num_pids
        self.K = num_imgs
        self.pid_index = defaultdict(list)
        for idx, (_, _, pid, _) in enumerate(dataset.samples):
            self.pid_index[pid].append(idx)
        self.pids = list(self.pid_index.keys())

    def __len__(self):
        return len(self.pids) * self.K

    def __iter__(self):
        random.shuffle(self.pids)
        batch = []
        for pid in self.pids:
            idxs    = self.pid_index[pid]
            replace = len(idxs) < self.K
            chosen  = (random.choices(idxs, k=self.K) if replace
                       else random.sample(idxs, self.K))
            batch.extend(chosen)
            if len(batch) == self.P * self.K:
                yield from batch
                batch = []

## 6. Evaluation

Rank-1 accuracy and mAP using the standard Market-1501 protocol. Optionally applies k-reciprocal re-ranking (Zhong et al. 2017) for ~5–10% mAP boost.

In [ ]:
@torch.no_grad()
def extract_features(model, loader, device, use_keypoints=False):
    """
    Returns:
        feats   : (N, D) L2-normalized BN embeddings
        pids    : (N,) integer person IDs
        cam_ids : (N,) integer camera IDs
        paths   : list[str]
    """
    model.eval()
    all_feats, all_pids, all_cams, all_paths = [], [], [], []

    for batch in tqdm(loader, desc="Extracting features", leave=False):
        imgs = batch["img"].to(device)
        kps  = batch["keypoints"].to(device) if use_keypoints else None
        feats = model(imgs, kps)
        all_feats.append(feats.cpu().numpy())
        all_pids.append(batch["pid"].numpy())
        all_cams.append(batch["cam"].numpy())
        all_paths.extend(batch["path"])

    return (
        np.concatenate(all_feats, axis=0),
        np.concatenate(all_pids,  axis=0),
        np.concatenate(all_cams,  axis=0),
        all_paths,
    )


def re_ranking(q_feats: np.ndarray,
               g_feats: np.ndarray,
               k1: int = 20, k2: int = 6,
               lambda_value: float = 0.3) -> np.ndarray:
    """
    k-reciprocal re-ranking (Zhong et al. 2017).
    Returns (Q, G) distance matrix — smaller means more similar.
    """
    query_num = q_feats.shape[0]
    all_num   = query_num + g_feats.shape[0]
    feat      = np.concatenate([q_feats, g_feats], axis=0).astype(np.float32)

    sq_norms = np.sum(feat ** 2, axis=1, keepdims=True)
    original_dist = sq_norms + sq_norms.T - 2.0 * (feat @ feat.T)
    original_dist = np.clip(original_dist, 0.0, None)

    col_max = original_dist.max(axis=0, keepdims=True)
    col_max[col_max == 0] = 1.0
    original_dist = original_dist / col_max

    initial_rank = np.argsort(original_dist, axis=1).astype(np.int32)

    V = np.zeros_like(original_dist, dtype=np.float32)
    half_k1 = int(np.around(k1 / 2))

    for i in range(all_num):
        fwd = initial_rank[i, : k1 + 1]
        bwd = initial_rank[fwd, : k1 + 1]
        recip = fwd[np.where(bwd == i)[0]]
        expansion = recip.copy()

        for cand in recip:
            cand_fwd = initial_rank[cand, : half_k1 + 1]
            cand_bwd = initial_rank[cand_fwd, : half_k1 + 1]
            cand_recip = cand_fwd[np.where(cand_bwd == cand)[0]]
            if (len(np.intersect1d(cand_recip, recip))
                    > (2.0 / 3.0) * len(cand_recip)):
                expansion = np.append(expansion, cand_recip)

        expansion = np.unique(expansion)
        w = np.exp(-original_dist[i, expansion])
        V[i, expansion] = w / w.sum()

    if k2 != 1:
        V_qe = np.zeros_like(V, dtype=np.float32)
        for i in range(all_num):
            V_qe[i] = V[initial_rank[i, :k2]].mean(axis=0)
        V = V_qe

    inv_index = [np.where(V[:, j] != 0)[0] for j in range(all_num)]

    jaccard_dist = np.zeros_like(original_dist, dtype=np.float32)
    for i in range(query_num):
        temp_min = np.zeros(all_num, dtype=np.float32)
        nonzero  = np.where(V[i] != 0)[0]
        for j in nonzero:
            idx = inv_index[j]
            temp_min[idx] += np.minimum(V[i, j], V[idx, j])
        jaccard_dist[i] = 1.0 - temp_min / (2.0 - temp_min + 1e-12)

    final_dist = jaccard_dist * (1.0 - lambda_value) + original_dist * lambda_value
    return final_dist[:query_num, query_num:]


def compute_ap(good_mask: np.ndarray) -> float:
    num_good = good_mask.sum()
    if num_good == 0:
        return 0.0
    positions  = np.where(good_mask)[0] + 1
    precisions = np.arange(1, num_good + 1) / positions
    return precisions.mean()


def evaluate(model, query_loader, gallery_loader, device,
             use_keypoints: bool = False,
             use_rerank:    bool = False,
             rerank_k1:     int  = 20,
             rerank_k2:     int  = 6,
             rerank_lambda: float = 0.3):
    """Returns (rank1 %, mAP %) using the Market-1501 protocol."""
    print("Extracting query features...")
    q_feats, q_pids, q_cams, _ = extract_features(model, query_loader, device, use_keypoints)
    print("Extracting gallery features...")
    g_feats, g_pids, g_cams, _ = extract_features(model, gallery_loader, device, use_keypoints)

    if use_rerank:
        print(f"Running re-ranking (k1={rerank_k1}, k2={rerank_k2}, λ={rerank_lambda})...")
        rerank_dist = re_ranking(q_feats, g_feats,
                                 k1=rerank_k1, k2=rerank_k2,
                                 lambda_value=rerank_lambda)
        score_matrix = -rerank_dist
    else:
        score_matrix = q_feats @ g_feats.T

    num_queries = q_feats.shape[0]
    rank1_hits  = 0
    ap_scores   = []

    for q_idx in range(num_queries):
        q_pid, q_cam = q_pids[q_idx], q_cams[q_idx]
        scores  = score_matrix[q_idx]
        ranked  = np.argsort(-scores)

        ranked_pids = g_pids[ranked]
        ranked_cams = g_cams[ranked]

        junk_mask  = (ranked_pids == q_pid) & (ranked_cams == q_cam)
        good_mask  = (ranked_pids == q_pid) & (ranked_cams != q_cam)
        good_valid = good_mask[~junk_mask]

        if good_valid.sum() == 0:
            continue

        rank1_hits += int(good_valid[0])
        ap_scores.append(compute_ap(good_valid))

    rank1 = 100.0 * rank1_hits / num_queries
    mAP   = 100.0 * float(np.mean(ap_scores)) if ap_scores else 0.0
    return rank1, mAP

## 7. Training Loop

In [ ]:
def train_one_epoch(model, loader,
                    id_loss_fn, tri_loss_fn, center_loss_fn,
                    center_loss_weight,
                    optimizer, optimizer_center,
                    device, epoch, use_keypoints):
    model.train()
    total_loss = total_id = total_tri = total_cen = correct = total = 0
    t0 = time.time()

    for step, batch in enumerate(loader):
        imgs      = batch["img"].to(device)
        labels    = batch["pid"].to(device)
        keypoints = batch["keypoints"].to(device) if use_keypoints else None

        logits, pool_feat, bn_feat = model(imgs, keypoints)

        loss_id  = id_loss_fn(logits, labels)
        loss_tri = tri_loss_fn(pool_feat, labels)
        loss_cen = center_loss_fn(pool_feat, labels)
        loss     = loss_id + loss_tri + center_loss_weight * loss_cen

        optimizer.zero_grad()
        optimizer_center.zero_grad()
        loss.backward()

        for p in center_loss_fn.parameters():
            if p.grad is not None:
                p.grad.data *= (1.0 / center_loss_weight)

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        optimizer_center.step()

        total_loss += loss.item()
        total_id   += loss_id.item()
        total_tri  += loss_tri.item()
        total_cen  += loss_cen.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

        if (step + 1) % 50 == 0:
            n = step + 1
            log.info(
                f"Epoch {epoch:3d} | step {n:4d}/{len(loader)} | "
                f"loss={total_loss/n:.4f} "
                f"(id={total_id/n:.4f}, tri={total_tri/n:.4f}, "
                f"cen={total_cen/n:.4f}) | "
                f"acc={100*correct/total:.1f}%"
            )

    n = len(loader)
    log.info(
        f"Epoch {epoch:3d} done [{time.time()-t0:.0f}s] | "
        f"avg loss={total_loss/n:.4f} | train acc={100*correct/total:.1f}%"
    )
    return total_loss / n

## 8. Setup: Data, Model, Optimizers

In [ ]:
def make_ds(split):
    return SkeletonReIDDataset(
        data_root         = DATA_ROOT,
        split             = split,
        transform         = build_transforms(split, IMG_SIZE, USE_RGB),
        use_rgb           = USE_RGB,
        use_keypoints     = USE_KEYPOINTS,
        skip_no_detection = USE_SKELETON and not INCLUDE_NO_DETECTION,
    )

train_ds = make_ds("train")
test_ds  = make_ds("test")
query_ds = make_ds("query")

num_classes = len(train_ds.pid2label)
log.info(f"Training identities: {num_classes} | train: {len(train_ds)} | "
         f"test: {len(test_ds)} | query: {len(query_ds)}")

sampler      = PKSampler(train_ds, NUM_PIDS, NUM_IMGS)
train_loader = DataLoader(
    train_ds, batch_size=NUM_PIDS * NUM_IMGS,
    sampler=sampler, num_workers=4, pin_memory=True, drop_last=True,
)
test_loader  = DataLoader(test_ds,  batch_size=EVAL_BS, shuffle=False, num_workers=4, pin_memory=True)
query_loader = DataLoader(query_ds, batch_size=EVAL_BS, shuffle=False, num_workers=4, pin_memory=True)

model = GaitReIDNet(
    num_classes   = num_classes,
    pretrained    = True,
    use_keypoints = USE_KEYPOINTS,
    use_ibn       = USE_IBN,
).to(device)
log.info(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

id_loss_fn     = LabelSmoothingCE(num_classes).to(device)
tri_loss_fn    = BatchHardTripletLoss().to(device)
center_loss_fn = CenterLoss(num_classes, feat_dim=model.feat_dim).to(device)

optimizer = optim.Adam([
    {"params": model.backbone.parameters(),   "lr": LR * 0.1},
    {"params": model.bottleneck.parameters(), "lr": LR},
    {"params": model.classifier.parameters(), "lr": LR},
], weight_decay=5e-4)

if USE_KEYPOINTS:
    kp_params = list(model.kp_encoder.parameters()) + list(model.fusion.parameters())
    optimizer.add_param_group({"params": kp_params, "lr": LR})

optimizer_center = optim.SGD(center_loss_fn.parameters(), lr=CENTER_LR)

if WARMUP_EPOCHS > 0:
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0,
                      total_iters=WARMUP_EPOCHS)
    cosine = CosineAnnealingLR(optimizer,
                               T_max  = EPOCHS - WARMUP_EPOCHS,
                               eta_min= 1e-7)
    scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[WARMUP_EPOCHS])
else:
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

## 9. Train

In [ ]:
best_rank1 = 0.0

for epoch in range(1, EPOCHS + 1):
    train_one_epoch(model, train_loader,
                    id_loss_fn, tri_loss_fn, center_loss_fn,
                    CENTER_WEIGHT,
                    optimizer, optimizer_center,
                    device, epoch, USE_KEYPOINTS)
    scheduler.step()

    if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
        rank1, mAP = evaluate(model, query_loader, test_loader,
                              device, USE_KEYPOINTS,
                              use_rerank    = USE_RERANK,
                              rerank_k1     = RERANK_K1,
                              rerank_k2     = RERANK_K2,
                              rerank_lambda = RERANK_LAMBDA)
        log.info(f"[Eval] Epoch {epoch:3d} — Rank-1: {rank1:.2f}%  mAP: {mAP:.2f}%")

        ckpt = os.path.join(OUTPUT_DIR, f"epoch_{epoch:03d}.pth")
        torch.save({
            "epoch":             epoch,
            "rank1":             rank1,
            "mAP":               mAP,
            "state_dict":        model.state_dict(),
            "optimizer":         optimizer.state_dict(),
            "optimizer_center":  optimizer_center.state_dict(),
            "center_loss_state": center_loss_fn.state_dict(),
        }, ckpt)

        if rank1 > best_rank1:
            best_rank1 = rank1
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pth"))
            log.info(f"  -> New best Rank-1: {rank1:.2f}%")

log.info(f"Training complete. Best Rank-1: {best_rank1:.2f}%")

## 10. Evaluate a Saved Checkpoint

Run this cell independently to evaluate any checkpoint without re-training.

In [ ]:
CHECKPOINT  = os.path.join(OUTPUT_DIR, "best_model.pth")  # change as needed
NUM_CLASSES = num_classes   # set manually if loading without the dataset above

eval_model = GaitReIDNet(
    num_classes   = NUM_CLASSES,
    pretrained    = False,
    use_keypoints = USE_KEYPOINTS,
    use_ibn       = USE_IBN,
).to(device)

ckpt  = torch.load(CHECKPOINT, map_location=device)
state = ckpt.get("state_dict", ckpt)
eval_model.load_state_dict(state)
print(f"Loaded: {CHECKPOINT}")

rank1, mAP = evaluate(
    eval_model, query_loader, test_loader, device,
    use_keypoints = USE_KEYPOINTS,
    use_rerank    = USE_RERANK,
    rerank_k1     = RERANK_K1,
    rerank_k2     = RERANK_K2,
    rerank_lambda = RERANK_LAMBDA,
)
print(f"\nRank-1: {rank1:.2f}%")
print(f"mAP   : {mAP:.2f}%")

## 11. Plot Training Curves

Parses `train.log` (the logging output captured to a file) and plots loss, accuracy, Rank-1, and mAP.

In [ ]:
EPOCH_RE = re.compile(
    r"Epoch\s+(\d+)\s+done\s+\[\d+s\]\s+\|\s+"
    r"avg loss=([\d.]+)\s+\|\s+"
    r"train acc=([\d.]+)%"
)
STEP_RE = re.compile(
    r"Epoch\s+(\d+)\s+\|\s+step\s+\d+/\d+\s+\|\s+"
    r"loss=[\d.]+\s+"
    r"\(id=([\d.]+),\s+tri=([\d.]+),\s+cen=([\d.]+)\)"
)
EVAL_RE = re.compile(
    r"\[Eval\]\s+Epoch\s+(\d+)\s+.*?Rank-1:\s+([\d.]+)%\s+mAP:\s+([\d.]+)%"
)


def parse_log(text: str):
    epochs, train_loss, train_acc     = [], [], []
    id_loss, tri_loss, cen_loss       = {}, {}, {}
    eval_epochs, eval_rank1, eval_map = [], [], []

    for line in text.splitlines():
        m = EPOCH_RE.search(line)
        if m:
            epochs.append(int(m.group(1)))
            train_loss.append(float(m.group(2)))
            train_acc.append(float(m.group(3)))
            continue
        m = STEP_RE.search(line)
        if m:
            ep = int(m.group(1))
            id_loss[ep]  = float(m.group(2))
            tri_loss[ep] = float(m.group(3))
            cen_loss[ep] = float(m.group(4))
            continue
        m = EVAL_RE.search(line)
        if m:
            eval_epochs.append(int(m.group(1)))
            eval_rank1.append(float(m.group(2)))
            eval_map.append(float(m.group(3)))

    return {
        "epochs":      epochs,
        "train_loss":  train_loss,
        "train_acc":   train_acc,
        "id_loss":     [id_loss.get(e)  for e in epochs],
        "tri_loss":    [tri_loss.get(e) for e in epochs],
        "cen_loss":    [cen_loss.get(e) for e in epochs],
        "eval_epochs": eval_epochs,
        "eval_rank1":  eval_rank1,
        "eval_map":    eval_map,
    }


def plot_training_curves(d: dict, out_path: str = "training_curves.png"):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    ax = axes[0, 0]
    ax.plot(d["epochs"], d["train_loss"], color="#21295C", linewidth=2.0, label="Total loss")
    ax.plot(d["epochs"], d["id_loss"],   color="#1C7293", linewidth=1.3, alpha=0.85, label="ID loss")
    ax.plot(d["epochs"], d["tri_loss"],  color="#5BA969", linewidth=1.3, alpha=0.85, label="Triplet loss")
    ax.set_title("Training loss", fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(loc="upper right", fontsize=9); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(d["epochs"], d["cen_loss"], color="#F2A65A", linewidth=2.0)
    ax.set_title("Center loss (log scale)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Center loss")
    ax.grid(True, alpha=0.3); ax.set_yscale("log")

    ax = axes[1, 0]
    ax.plot(d["epochs"], d["train_acc"], color="#065A82", linewidth=2.0)
    ax.set_title("Train accuracy (closed-set, 751 IDs)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 102); ax.grid(True, alpha=0.3)
    ax.axhline(99, color="#888", linestyle=":", linewidth=1, alpha=0.6)

    ax = axes[1, 1]
    ax.plot(d["eval_epochs"], d["eval_rank1"], "-o", color="#065A82",
            linewidth=2.0, markersize=5, label="Rank-1")
    ax.plot(d["eval_epochs"], d["eval_map"],   "-s", color="#C45A5A",
            linewidth=2.0, markersize=5, label="mAP")
    ax.set_title("Eval Rank-1 & mAP (open-set, 750 IDs)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Score (%)")
    ax.set_ylim(0, 100); ax.legend(loc="lower right", fontsize=10); ax.grid(True, alpha=0.3)

    if d["eval_rank1"]:
        best_idx = int(max(range(len(d["eval_rank1"])), key=lambda i: d["eval_rank1"][i]))
        bx, by = d["eval_epochs"][best_idx], d["eval_rank1"][best_idx]
        ax.annotate(f"Best: {by:.2f}% @ epoch {bx}",
                    xy=(bx, by), xytext=(bx - 30, by - 18), fontsize=9, color="#065A82",
                    arrowprops=dict(arrowstyle="->", color="#065A82", alpha=0.7))

    fig.suptitle("ReID training run — 120 epochs, RGB primary input",
                 fontsize=15, fontweight="bold", y=1.00)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()


# --- Run ---
log_text = Path(LOG_PATH).read_text()
d = parse_log(log_text)
print(f"Parsed {len(d['epochs'])} epochs, {len(d['eval_epochs'])} evaluations.")
if d["train_acc"]:
    print(f"Final train acc: {d['train_acc'][-1]:.2f}%  |  "
          f"Final Rank-1: {d['eval_rank1'][-1]:.2f}%  |  "
          f"Final mAP: {d['eval_map'][-1]:.2f}%")
plot_training_curves(d, out_path="training_curves.png")